In [ ]:
import base64
import time
from pathlib import Path
from ollama import chat

IMAGE_PATH = r"C:\Users\RDPprestauser1\Pictures\Prenom_Fdr_1.png"

# ── Modèles à tester ──
MODELS = [
    "qwen2.5vl:3b",
    "qwen2.5vl:7b",
    "gemma3:4b",
    "llama3.2-vision:11b",
    "minicpm-v",
    "deepseek-ocr",
]

PROMPT = """Look at this image carefully. There is handwritten text on it.
Read the handwritten text and write it out exactly as you see it.
Your answer must contain ONLY the text you read, nothing else."""


def test_model(model_name: str, image_path: str) -> dict:
    """Teste un modèle et retourne le résultat + temps."""
    print(f"\n{'='*50}")
    print(f"Test : {model_name}")
    print(f"{'='*50}")

    start = time.time()
    try:
        response = chat(
            model=model_name,
            messages=[
                {
                    "role": "user",
                    "content": PROMPT,
                    "images": [image_path],   # Ollama accepte directement le chemin
                }
            ],
            options={
                "temperature": 0.0,
                "num_predict": 100,
            },
        )
        elapsed = time.time() - start
        result = response.message.content.strip()

        print(f"  Résultat : {result}")
        print(f"  Temps    : {elapsed:.1f}s")

        return {"model": model_name, "result": result, "time": elapsed, "error": None}

    except Exception as e:
        elapsed = time.time() - start
        print(f"  ERREUR   : {e}")
        return {"model": model_name, "result": None, "time": elapsed, "error": str(e)}


def main():
    img = Path(IMAGE_PATH)
    if not img.exists():
        raise FileNotFoundError(f"Image introuvable : {IMAGE_PATH}")

    results = []
    for model in MODELS:
        res = test_model(model, IMAGE_PATH)
        results.append(res)

    # ── Tableau récapitulatif ──
    print(f"\n{'='*60}")
    print(f"{'MODÈLE':<25} {'TEMPS':>8} {'RÉSULTAT'}")
    print(f"{'='*60}")
    for r in results:
        status = r["result"] if r["result"] else f"ERREUR: {r['error']}"
        print(f"{r['model']:<25} {r['time']:>6.1f}s  {status}")


if __name__ == "__main__":
    main()